Primero ubicamos el archivo a cargar en nuestro dataset:

In [3]:
url="https://raw.githubusercontent.com/ccastaneda-boot/Parcial4-CarlosCasta-eda-1743602001/refs/heads/main/data/clave_D_asociacion.csv"

Importamos librerias a utilziar para el análisis:

In [1]:
import pandas as pd
import numpy as np
from itertools import combinations

In [ ]:
Ahora cargamos en nuestro dataset y lo visualizamos:

In [4]:
df=pd.read_csv(url)
df.head()

,transaccion_id,cliente_id,fecha,categoria,item,cantidad,canal
0,D-T0001,D-C0015,2026-04-02,Bebidas,Chocolate,1,Telefono
1,D-T0001,D-C0015,2026-04-02,Extras,Leche_extra,1,Telefono
2,D-T0001,D-C0015,2026-04-02,Salados,Wrap,1,Telefono
3,D-T0002,D-C0079,2026-01-21,Bebidas,Cafe_americano,1,Web
4,D-T0002,D-C0079,2026-01-21,Salados,Ensalada,1,Web


Verificamos su estructura y vemos primeras filas del dataset:

In [8]:

print("Dimensiones del dataset:")
print(df.shape)

print("\nColumnas del dataset:")
print(df.columns)


print("\nInformación general del dataset")
print(df.info())

print("\nPrimeras filas del dataset:")
print(df.head())


Dimensiones del dataset:
(607, 7)

Columnas del dataset:
Index(['transaccion_id', 'cliente_id', 'fecha', 'categoria', 'item',
       'cantidad', 'canal'],
      dtype='object')

Información general del dataset
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 607 entries, 0 to 606
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   transaccion_id  607 non-null    object
 1   cliente_id      607 non-null    object
 2   fecha           607 non-null    object
 3   categoria       607 non-null    object
 4   item            607 non-null    object
 5   cantidad        607 non-null    int64 
 6   canal           606 non-null    object
dtypes: int64(1), object(6)
memory usage: 33.3+ KB
None

Primeras filas del dataset:
  transaccion_id cliente_id       fecha categoria            item  cantidad  \
0        D-T0001    D-C0015  2026-04-02   Bebidas       Chocolate         1   
1        D-T0001    D-C0015  2026-04-02    Extr

Verificamos valores nulos y duplicados:

In [9]:
print("\nValores nulos por columna:")
print(df.isnull().sum())


Valores nulos por columna:
transaccion_id    0
cliente_id        0
fecha             0
categoria         0
item              0
cantidad          0
canal             1
dtype: int64


Vamos a rellenar el valor nulo en 'canal' para que no cause problemas

In [30]:
df['canal'] = df['canal'].fillna('Desconocido')

In [10]:
print("\nCantidad de registros duplicados")
print(df.duplicated().sum())


Cantidad de registros duplicados
1


In [19]:
print("\nRegistros duplicados:")
print(df[df.duplicated(keep=False)])


Registros duplicados:
    transaccion_id cliente_id       fecha   categoria       item  cantidad  \
15         D-T0006    D-C0029  2026-04-27  Reposteria  Croissant         1   
606        D-T0006    D-C0029  2026-04-27  Reposteria  Croissant         1   

    canal  
15    Web  
606   Web  


Vemos la estadística Descriptiva general:

In [22]:
print("\nEstadística descriptiva:")
print(df.describe())


Estadística descriptiva:
         cantidad
count  607.000000
mean     1.459638
std      0.752072
min      1.000000
25%      1.000000
50%      1.000000
75%      2.000000
max      4.000000


Revisamos valores atípicos

In [25]:
variables_numericas = [ "cantidad"]

print("\nRevisión de valores atípicos:")
for columna in variables_numericas:
    media = df[columna].mean()
    desviacion = df[columna].std()

    limite_inferior = media - 2 * desviacion
    limite_superior = media + 2 * desviacion

    outliers = df[
        (df[columna] < limite_inferior) |
        (df[columna] > limite_superior)
    ]

    print("Variable:", columna)
    print("Media:", round(media, 2))
    print("Desviación estándar:", round(desviacion, 2))
    print("Límite inferior:", round(limite_inferior, 2))
    print("Límite superior:", round(limite_superior, 2))
    print("Cantidad de valores atípicos:", len(outliers))
    print("-" * 40)


Revisión de valores atípicos:
Variable: cantidad
Media: 1.46
Desviación estándar: 0.75
Límite inferior: -0.04
Límite superior: 2.96
Cantidad de valores atípicos: 64
----------------------------------------


Revisamos variables categóricas:

In [23]:
print("\nDistribución de Canal:")
print(df["canal"].value_counts())

print("\nDistribución de Categoria:")
print(df["categoria"].value_counts())

print("\nDistribución de Item:")
print(df["item"].value_counts())



Distribución de Canal:
canal
Tienda      271
Web         167
App         138
Telefono     30
Name: count, dtype: int64

Distribución de Categoria:
categoria
Bebidas       158
Reposteria    151
Salados       110
Combos        110
Extras         78
Name: count, dtype: int64

Distribución de Item:
item
Galleta             49
Cafe_americano      48
Latte               48
Muffin              43
Croissant           42
Sandwich            41
Combo_estudiante    41
Capuchino           40
Hielo               31
Combo_oficina       24
Ensalada            24
Combo_desayuno      24
Panini              23
Chocolate           22
Wrap                22
Combo_tarde         21
Sirope              18
Leche_extra         17
Dona                17
Crema_batida        12
Name: count, dtype: int64


Preparamos los campos para aplicar la regla de asociación:


Instalamos la librería mlxtend.

In [33]:
pip install mlxtend

Ahora, vamos a transformar los datos para que cada fila represente una transacción y cada columna un ítem. Usaremos un enfoque de codificación 'one-hot' donde 1 indica la presencia del ítem en la transacción y 0 su ausencia.

In [34]:
from mlxtend.frequent_patterns import apriori, association_rules

# Agrupar ítems por transaccion_id y item, y luego pivotear para tener ítems como columnas
basket = df.groupby(['transaccion_id', 'item'])['cantidad'].sum().unstack().reset_index().fillna(0).set_index('transaccion_id')

# Convertir las cantidades a formato booleano (presente/ausente)
def encode_units(x):
    if x <= 0: # Si la cantidad es 0 o menos, el ítem no está presente
        return 0
    if x >= 1: # Si la cantidad es 1 o más, el ítem está presente
        return 1

basket_sets = basket.applymap(encode_units)

print("Primeras 5 filas del dataset de transacciones (formato one-hot):")
display(basket_sets.head())

Primeras 5 filas del dataset de transacciones (formato one-hot):


/tmp/ipykernel_7036/827726169.py:13: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  basket_sets = basket.applymap(encode_units)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


item,Cafe_americano,Capuchino,Chocolate,Combo_desayuno,Combo_estudiante,Combo_oficina,Combo_tarde,Crema_batida,Croissant,Dona,Ensalada,Galleta,Hielo,Latte,Leche_extra,Muffin,Panini,Sandwich,Sirope,Wrap
transaccion_id,,,,,,,,,,,,,,,,,,,,
D-T0001,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1
D-T0002,1,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0
D-T0003,0,1,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0
D-T0004,0,1,1,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,1,0
D-T0005,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Con los datos preparados, aplicamos el algoritmo Apriori para encontrar los conjuntos de ítems frecuentes. Luego, generaremos las reglas de asociación usando estas frecuencias.

In [35]:
# Encontrar los conjuntos de ítems frecuentes con un soporte mínimo
frequent_itemsets = apriori(basket_sets, min_support=0.07, use_colnames=True)

# Generar las reglas de asociación
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.5)

print("Conjuntos de ítems frecuentes (primeras 5 filas):")
display(frequent_itemsets.head())

print("\nReglas de asociación generadas (primeras 5 filas):")
display(rules.head())

Conjuntos de ítems frecuentes (primeras 5 filas):


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

,support,itemsets
0,0.266667,(Cafe_americano)
1,0.222222,(Capuchino)
2,0.122222,(Chocolate)
3,0.133333,(Combo_desayuno)
4,0.227778,(Combo_estudiante)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)



Reglas de asociación generadas (primeras 5 filas):


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,(Sandwich),(Cafe_americano),0.227778,0.266667,0.138889,0.609756,2.286585,1.0,0.078148,1.879167,0.728633,0.390625,0.467849,0.565295
1,(Cafe_americano),(Sandwich),0.266667,0.227778,0.138889,0.520833,2.286585,1.0,0.078148,1.611594,0.767273,0.390625,0.379496,0.565295
2,(Croissant),(Capuchino),0.227778,0.222222,0.116667,0.512195,2.304878,1.0,0.066049,1.594444,0.733128,0.350000,0.372822,0.518598
3,(Capuchino),(Croissant),0.222222,0.227778,0.116667,0.525000,2.304878,1.0,0.066049,1.625731,0.727891,0.350000,0.384892,0.518598
4,(Galleta),(Combo_estudiante),0.272222,0.227778,0.150000,0.551020,2.419114,1.0,0.087994,1.719949,0.806050,0.428571,0.418588,0.604778


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Las reglas de asociación resultantes muestran la relación entre los ítems. Podemos ordenar estas reglas por confianza o por soporte para identificar las asociaciones más fuertes o más comunes.

In [36]:
# Ordenar las reglas por confianza descendente
rules_sorted_by_confidence = rules.sort_values(by='confidence', ascending=False)

print("Reglas de asociación ordenadas por confianza (primeras 10 filas):")
display(rules_sorted_by_confidence.head(10))

# Ordenar las reglas por lift descendente
rules_sorted_by_lift = rules.sort_values(by='lift', ascending=False)

print("\nReglas de asociación ordenadas por lift (primeras 10 filas):")
display(rules_sorted_by_lift.head(10))

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Reglas de asociación ordenadas por confianza (primeras 10 filas):


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
5,(Combo_estudiante),(Galleta),0.227778,0.272222,0.150000,0.658537,2.419114,1.0,0.087994,2.131349,0.759659,0.428571,0.530814,0.604778
0,(Sandwich),(Cafe_americano),0.227778,0.266667,0.138889,0.609756,2.286585,1.0,0.078148,1.879167,0.728633,0.390625,0.467849,0.565295
4,(Galleta),(Combo_estudiante),0.272222,0.227778,0.150000,0.551020,2.419114,1.0,0.087994,1.719949,0.806050,0.428571,0.418588,0.604778
6,(Muffin),(Latte),0.238889,0.266667,0.127778,0.534884,2.005814,1.0,0.064074,1.576667,0.658838,0.338235,0.365751,0.507025
3,(Capuchino),(Croissant),0.222222,0.227778,0.116667,0.525000,2.304878,1.0,0.066049,1.625731,0.727891,0.350000,0.384892,0.518598
1,(Cafe_americano),(Sandwich),0.266667,0.227778,0.138889,0.520833,2.286585,1.0,0.078148,1.611594,0.767273,0.390625,0.379496,0.565295
2,(Croissant),(Capuchino),0.227778,0.222222,0.116667,0.512195,2.304878,1.0,0.066049,1.594444,0.733128,0.350000,0.372822,0.518598



Reglas de asociación ordenadas por lift (primeras 10 filas):


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
4,(Galleta),(Combo_estudiante),0.272222,0.227778,0.150000,0.551020,2.419114,1.0,0.087994,1.719949,0.806050,0.428571,0.418588,0.604778
5,(Combo_estudiante),(Galleta),0.227778,0.272222,0.150000,0.658537,2.419114,1.0,0.087994,2.131349,0.759659,0.428571,0.530814,0.604778
2,(Croissant),(Capuchino),0.227778,0.222222,0.116667,0.512195,2.304878,1.0,0.066049,1.594444,0.733128,0.350000,0.372822,0.518598
3,(Capuchino),(Croissant),0.222222,0.227778,0.116667,0.525000,2.304878,1.0,0.066049,1.625731,0.727891,0.350000,0.384892,0.518598
0,(Sandwich),(Cafe_americano),0.227778,0.266667,0.138889,0.609756,2.286585,1.0,0.078148,1.879167,0.728633,0.390625,0.467849,0.565295
1,(Cafe_americano),(Sandwich),0.266667,0.227778,0.138889,0.520833,2.286585,1.0,0.078148,1.611594,0.767273,0.390625,0.379496,0.565295
6,(Muffin),(Latte),0.238889,0.266667,0.127778,0.534884,2.005814,1.0,0.064074,1.576667,0.658838,0.338235,0.365751,0.507025


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

**INTERPRETACIÓN DE 5 REGLAS DEL NEGOCIO:**

1. galleta->combo_estudiante: es muy probable que si un cliente compra una galleta, tambien compre un combo_estudiante
2. combo_estudiante->galleta: Es muy probable que cuando un cliente compre un combo_estudiante, tambien compre una galleta. Esto sugiere que la galleta es un complemento que gusta cuando se compra el combo_estudiante y refuerza la regla 1
3. Croissant->Capuchino: Es mas frecuente que cuando se venda un Croissant, se venda junto con un capuchino.
4. sandwich-> Cafe_americano: Es mas frecuente que cuando se venda un sandwich, se venda un café americano. Esto suele ser una combinación clásica.
5. muffin->Latte: Es mas frecuente que cuando se venda un muffin, se venda un latte. puede  ser una buena promoción.


**RECOMENDACIONES:**

1. Al colocar los productos en mostrador, buscar asegurar que estén cerca estas asociaciones detectadas en las reglas (ej. Junto a la oferta del café, colocar el sandwich).

2. Buscar crear combos que contengas las asociaciones detectadas en estas reglas, ej. "Combo de Muffin + latte por $3.00" o por la compra de un muffin, llevate un latte por $0.50 adicionales

3. Capacitar al personal para que realice sugerencias a los clientes de venta cruzada. Por ejemplo, si un clinete pide un Croissant, el vendedor le sugiere un capuchino.